In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
data = pd.read_csv("../reviews_filtered.csv")
 # import reviews

print(data.head(10))

   Unnamed: 0  movie_id                       name  count_  user_id  rating
0           0     34222  slumber-party-massacre-ii      32   438333       7
1           1     34222  slumber-party-massacre-ii      32   437213       8
2           2     34222  slumber-party-massacre-ii      32   434710       7
3           3     34222  slumber-party-massacre-ii      32   423063       5
4           4     34222  slumber-party-massacre-ii      32   244269       9
5           5     34222  slumber-party-massacre-ii      32   188272       7
6           6     34222  slumber-party-massacre-ii      32    30514       6
7           7     34222  slumber-party-massacre-ii      32     8026      10
8           8    462065         run-sweetheart-run      32  1146451       5
9           9    462065         run-sweetheart-run      32  1067317       7


import re

love_death_robots_filter = re.compile("love-death-robots")
black_mirror_filter = re.compile("black-mirror")

to_be_removed = []

for idx, row in data.iterrows():
    if (re.match(love_death_robots_filter, row['name'])):
        to_be_removed.append(idx)
    elif (re.match(black_mirror_filter, row['name'])):
        to_be_removed.append(idx)

data = data.drop(to_be_removed)

data.to_csv("../reviews_filtered.csv")

In [3]:
unique_movies = sorted(list(set(zip(data['movie_id'], data['name']))))
unique_users = sorted(list(set(data['user_id'])))

num_movies = len(unique_movies)
num_users = len(unique_users)

print("%d Unique Movies" % num_movies)
print("%d Unique Users" % num_users)

5937 Unique Movies
2516 Unique Users


In [4]:
# these are a set of variables to make conversion between labels and attribute lookup easier
# users have two labels: their ids in the original dataset and their idxs in the list from the previous cell
# movies have 3: the same two as users, and also a name

movie_id_to_idx = {movie_id: idx for idx,
                   (movie_id, _) in enumerate(unique_movies)}
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(unique_users)}

movie_name_to_idx = {movie_name: idx for idx,
                     (_, movie_name) in enumerate(unique_movies)}

movie_names = [movie_name for (_, movie_name) in unique_movies]

movie_id_to_name = {movie_id: movie_name for (
    movie_id, movie_name) in unique_movies}

# we also add a variable to easily look up how many reviews a movie has in our dataset
movie_freqs = {movie_id_to_idx[movie_id]: freq for (
    movie_id, freq) in zip(data['movie_id'], data['count_'])}


In [5]:
# loos up id and list idx for a movie given its name
# this uses fuzzy string matching so we can look up the attributes for a given movie
def look_up_movie(movie_name):
    match_string = re.compile(".*" + movie_name + ".*")
    for idx, (movie_id, movie_name) in enumerate(unique_movies):
        # movie_name = movie_id_to_name[movie_id]
        if (re.match(match_string, movie_name)):
            yield (movie_name, movie_id, idx, movie_freqs[idx])

In [6]:
# tests the above function
for (movie_name, movie_id, idx, freq) in look_up_movie("emma"):
    print(movie_name, movie_id, idx, freq)

emma-2020 485212 4103 454
the-social-dilemma 581924 4541 191


In [7]:
from sklearn.model_selection import train_test_split
from surprise import Dataset, NormalPredictor, Reader


reader = Reader(rating_scale=(1, 10))
# the reader class in surprise just parses ratings on a given scale
# ours are from 1 to 10

dataset = Dataset.load_from_df(data[["user_id", "movie_id", "rating"]], reader)
# load data into surprise dataset class

In [8]:
from surprise import accuracy, Dataset, SVD
from surprise.model_selection import KFold

# tests the svd algorithm on the dataset as a baseline using 3 fold cross validation

# define a cross-validation iterator
kf = KFold(n_splits=3)

algo = SVD()

for trainset, testset in kf.split(dataset):

    # train and test svd algorithm.
    algo.fit(trainset)
    predictions = algo.test(testset)

    # Compute and print Root Mean Squared Error
    accuracy.rmse(predictions, verbose=True)

RMSE: 1.5385
RMSE: 1.5374
RMSE: 1.5409
